####1. Read data from the apache-logs.txt file


In [0]:
%sql
SELECT *
FROM read_files("/Volumes/dev/spark_db/datasets/spark_programming/data/apache-logs.txt" , 
format=> 'text')

In [0]:
text_df = spark.read.format('text').load("/Volumes/dev/spark_db/datasets/spark_programming/data/apache-logs.txt" )

In [0]:
text_df.display()

####2. Develop a strategy to extract the following fields
1. ip_address: It is the IP address of the site visitor.
2. visit_timestamp: It is the date and time of the site visit. Parse and format the timestamp to YYYY-MM-DD HH:MI:SS Z
3. visit_resource: Which resource from our website was accessed
4. referring_url: It is the clean URL of the referring website.


Develop a regular expression

In [0]:
%sql
SELECT regexp_extract(value, '^\\s*(\\d+\\.\\d+\\.\\d+\\.\\d+)', 1) AS ip_address ,

to_timestamp(regexp_extract(value,'\\[(.*?)\\]\\s*',1) ,"dd/MMM/yyyy:HH:mm:ss Z") AS visit_timestamp,

regexp_extract(value, '\\"GET(.*?)\\s*HTTP.*"',1) AS visit_resource,

regexp_extract(value, '\\d{3}\\s*\\d+?\\s*\\"(http.*?)\\"',1) AS resource_url


FROM read_files("/Volumes/dev/spark_db/datasets/spark_programming/data/apache-logs.txt" , 
format=> 'text')

Alternative approach : Preparing a prompt to pass on to LLM model 


In [0]:
prompt = """
You will be provided with an Apache log file record. It is an unstructured text record. 
Each record represents some information for our website visits, such as what is the IP address of the visitor, 
What is the date and time of the visit, which resource was requested, and the URL of the referring website? 
You are asked to parse the log file record and extract the following fields.
ip_address: It is the IP address of the site visitor.
visit_timestamp: It is the date and time of the site visit. Parse and format the timestamp to YYYY-MM-DD HH:MI:SS Z
visit_resource: Which resource from our website was accessed?
referring_url: It is the clean URL of the referring website. When the actual referring URL is not given, 
you can extract the URL from the user agent. For cleaning the URL, you should take the values only up to the domain extension, 
such as .com, .in, .uk, etc.
Give only the final answer in the JSON format.
Record:
"""

In [0]:
prompt

Develop an AI query function --> prebuilt by databricks to be used in Apache spark df.this function calls the LLM provided by databricks.

In [0]:
# create column that concats prompt and value for each row so you can pass that to LLM function for getting the output

In [0]:
from pyspark.sql.functions import concat, lit, col,expr

text_ai_query_df = (text_df.limit(10).withColumn('input_prompt', concat(lit(prompt),col('value')))
                    .withColumn('json_extract', expr("""
                                                     ai_query( 
                                                     endpoint=> "databricks-llama-4-maverick",
                                                     request=> input_prompt,
                                                     responseFormat => 'struct<extract:struct<
                                                     ip_address : string,
                                                     visit_timestamp : string,
                                                     visit_resource : string,
                                                     referring_url : string
                                                     >>')
                                                     """
                    )))

In [0]:
display(text_ai_query_df)

Parse json_extract to individual fields

In [0]:
from pyspark.sql.functions import from_json
text_ai_ouput_json_parse = (text_ai_query_df.withColumn(
                    "json_extract" , from_json(col("json_extract"), 
                                               ("ip_address STRING, visit_timestamp STRING, visit_resource STRING,referring_url STRING ")
                                               )    
)
)



In [0]:
text_ai_ouput_json_parse.display()

In [0]:
text_ai_ouput_json_parse_expansion = text_ai_ouput_json_parse.selectExpr("json_extract.*")

In [0]:
from pyspark.sql.functions import to_timestamp
final_df = text_ai_ouput_json_parse_expansion.withColumn("visit_timestamp", to_timestamp(col("visit_timestamp"), "yyyy-MM-dd HH:mm:ss Z"))

final_df.display()